# nodes.llm_filter_candidates

Run LLM negative selection to filter cosine candidates.

For each job ad, builds a prompt with the ad's context (title, category,
description excerpt) and its top-N candidate occupations from cosine matching.
The LLM identifies which candidates to DROP, keeping the functional matches.

Reads from `cosine_candidates/candidates.parquet` and produces both:
- `filtered_matches.parquet`: kept candidates with cosine_score
- `dropped_matches.parquet`: dropped candidates with cosine_score and drop reason

Node variables:
- `llm_model` (global): Model key from llm_models.toml
- `llm_batch_size` (global): Number of prompts per LLM call
- `llm_max_new_tokens` (global): Max tokens per LLM response
- `max_concurrent_chunks` (per-node): Max concurrent batch LLM calls
- `filter_resume` (per-node): Resume from previous partial run
- `filter_max_retries` (per-node): Retry rounds for failed ads
- `system_prompt` (per-node): Path in config/prompt_library/
- `user_prompt` (per-node): Path in config/prompt_library/
- `run_name` (global): Pipeline run name

In [ ]:
#|default_exp llm_filter_candidates
#|export_as_func true

In [ ]:
#|top_export
import json
from typing import List

from pydantic import BaseModel, field_validator

class FilterResponseModel(BaseModel):
    keep: List[int]

    @field_validator("keep")
    @classmethod
    def keep_indices_positive(cls, v):
        if len(v) < 1:
            raise ValueError("must keep at least 1 candidate")
        for idx in v:
            if idx < 1:
                raise ValueError(f"keep indices must be 1-based positive integers, got {idx}")
        return v

In [ ]:
#|set_func_signature
async def main(ctx, print, ad_ids: list[int]) -> {
    'successful_ad_ids': list[int]
}:
    """Run LLM negative selection to filter cosine candidates."""
    ...


Retrieve input arguments

In [ ]:
from dev_utils import *
run_name = 'test_local'
set_node_func_args('llm_filter_candidates', run_name=run_name)
show_node_vars('llm_filter_candidates', run_name=run_name)


# Function body

## Read node variables

In [ ]:
#|export
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

from ai_index import const
from ai_index.utils import (
    ResultStore, run_batched, load_prompt, allm_generate,
    extract_json, is_reasoning_model, uses_structured_output,
)
from isambard_utils.orchestrate import StagedInput
from isambard_utils.staging import astage_files

In [ ]:
#|export
run_name = ctx.vars["run_name"]
llm_model = ctx.vars["llm_model"]
sbatch_cache = ctx.vars["sbatch_cache"]
sbatch_time = ctx.vars["sbatch_time"]
batch_size = ctx.vars["llm_batch_size"]
max_new_tokens = ctx.vars["llm_max_new_tokens"]
temperature = ctx.vars["temperature"]
top_p = ctx.vars["top_p"]
top_k = ctx.vars["top_k"]
max_concurrent = ctx.vars["max_concurrent_chunks"]
resume = ctx.vars["filter_resume"]
max_retries = ctx.vars["filter_max_retries"]
raise_on_failure = ctx.vars["filter_raise_on_failure"]
duckdb_memory_limit = ctx.vars["duckdb_memory_limit"]

_is_reasoning = is_reasoning_model(llm_model)
_use_structured_output = uses_structured_output(llm_model)

_system_prompt_key = ctx.vars["system_prompt"]
_user_prompt_key = ctx.vars["user_prompt"]
if not _use_structured_output:
    suffix = "_reasoning" if _is_reasoning else "_unstructured"
    _system_prompt_key += suffix
    _user_prompt_key += suffix

SYSTEM_PROMPT = load_prompt(_system_prompt_key)
USER_PROMPT_TEMPLATE = load_prompt(_user_prompt_key)

output_dir = const.pipeline_store_path / run_name / "llm_filter_candidates"
output_dir.mkdir(parents=True, exist_ok=True)
db_path = output_dir / "filter_results.duckdb"

## Stage data and build candidate index

Stage candidates, ad texts, and O*NET descriptions to Isambard once.
Read candidate structure into a sorted Arrow table with offset index
so per-chunk lookups are O(1) dict access instead of DuckDB queries.

In [ ]:
#|export
matches_path = const.pipeline_store_path / run_name / "cosine_candidates" / "candidates.parquet"
ad_texts_path = const.pipeline_store_path / run_name / "sample_ads" / "ad_texts.parquet"

# Build O*NET candidate text for the LLM prompt: description + top 5 tasks.
# No alternate titles (they blow up the context without helping the LLM's
# keep/drop decision; the O*NET title is already shown separately).
_onet_targets = pd.read_parquet(const.onet_targets_path)

def _build_onet_text(row):
    parts = [row["Description"]]
    tasks = row["Top_Tasks"]
    if len(tasks) > 0:
        parts.append("Key tasks: " + "; ".join(tasks[:5]))
    return " ".join(parts)

_onet_descriptions = dict(zip(_onet_targets["O*NET-SOC Code"], _onet_targets.apply(_build_onet_text, axis=1)))

# Export onet_descriptions.json for staging
staging_dir = output_dir / "_staging"
staging_dir.mkdir(parents=True, exist_ok=True)
onet_desc_path = staging_dir / "onet_descriptions.json"
with open(onet_desc_path, "w") as f:
    json.dump(_onet_descriptions, f, sort_keys=True)

# Stage files to Isambard
print("llm_filter: staging files to Isambard...")
staged_refs = await astage_files(
    {
        "candidates": matches_path,
        "ad_texts": ad_texts_path,
        "onet_descriptions": onet_desc_path,
    },
    print_fn=print,
)
print("llm_filter: staging complete")

print(f"llm_filter: {len(ad_ids)} ads to process")
print(f"llm_filter: reading candidates from {const.rel(matches_path)}")

# Read candidate structure ONCE as sorted Arrow table with offset index.
# At 5M ads x 20 candidates, this is ~500MB columnar vs ~15GB as Python dicts.
print("llm_filter: reading candidate structure...")
_all_matches = pq.read_table(
    matches_path,
    columns=["ad_id", "onet_code", "onet_title", "cosine_score"],
)
_sort_indices = pc.sort_indices(_all_matches, sort_keys=[("ad_id", "ascending")])
candidates_table = _all_matches.take(_sort_indices)
del _all_matches, _sort_indices

# Build offset index: ad_id -> (start, end) row range in sorted table
_ad_ids_arr = candidates_table.column("ad_id").to_numpy()
candidates_index = {}
_i = 0
while _i < len(_ad_ids_arr):
    _aid = int(_ad_ids_arr[_i])
    _start = _i
    while _i < len(_ad_ids_arr) and _ad_ids_arr[_i] == _aid:
        _i += 1
    candidates_index[_aid] = (_start, _i)
del _ad_ids_arr
print(f"  {len(candidates_index)} ads with candidates, {len(candidates_table)} rows ({candidates_table.nbytes / 1e6:.0f} MB)")

## Define work function

For each chunk of ad IDs, build prompts and call the LLM.

In [ ]:
#|export
def _validate_response(raw: str, n_candidates: int) -> str | None:
    """Validate an LLM filter response. Returns None if valid, or an error string."""
    try:
        parsed = FilterResponseModel.model_validate_json(raw)
        # Check indices are in valid range
        for idx in parsed.keep:
            if idx < 1 or idx > n_candidates:
                return f"keep index {idx} out of range [1, {n_candidates}]"
        return None
    except Exception as e:
        return f"{type(e).__name__}: {e}"


_slurm_jobs = []

async def _work_fn(chunk_ids):
    """Build StagedInput, call LLM, validate responses, return DataFrame."""
    # Pre-compute n_candidates_per_ad from offset index (no DuckDB query)
    n_candidates_per_ad = []
    for ad_id in chunk_ids:
        start, end = candidates_index[ad_id]
        n_candidates_per_ad.append(end - start)

    # The resolver on Isambard reads staged parquets/JSON and builds
    # the formatted prompts for this chunk using predicate pushdown.
    staged_prompts = StagedInput(
        resolver="filter_candidates_prompts",
        sources=staged_refs,
        params={
            "ad_ids": list(chunk_ids),
            "user_prompt_template": USER_PROMPT_TEMPLATE,
        },
    )

    _sa = {}
    schema = FilterResponseModel.model_json_schema() if _use_structured_output else None
    responses = await allm_generate(
        staged_prompts,
        model=llm_model,
        system_message=SYSTEM_PROMPT,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        json_schema=schema,
        cache=sbatch_cache,
        time=sbatch_time,
        slurm_accounting=_sa,
    )
    if _sa: _slurm_jobs.append(_sa)

    records = []
    for ad_id, response, n_cands in zip(chunk_ids, responses, n_candidates_per_ad):
        if _is_reasoning or not _use_structured_output:
            parsed = extract_json(response, validator=FilterResponseModel.model_validate)
            if parsed is None:
                records.append({"id": ad_id, "data": response, "error": "Failed to extract valid JSON from model output"})
                continue
            response = json.dumps(parsed)
        error = _validate_response(response, n_cands)
        records.append({"id": ad_id, "data": response, "error": error})
    return pd.DataFrame(records)

## Run batched LLM calls

In [ ]:
#|export
store = ResultStore(db_path, {
    "id": "BIGINT NOT NULL",
    "data": "VARCHAR NOT NULL",
    "error": "VARCHAR",
}, memory_limit=duckdb_memory_limit)

filter_meta = await run_batched(
    ad_ids, store, _work_fn,
    batch_size=batch_size,
    max_concurrent=max_concurrent,
    max_retries=max_retries,
    resume=resume,
    node_name="llm_filter_candidates",
    print_fn=print,
    raise_on_failure=raise_on_failure,
)
store.close()
del store
filter_meta["slurm_jobs"] = _slurm_jobs
filter_meta["slurm_total_seconds"] = sum(j.get("elapsed_seconds", 0) for j in _slurm_jobs)
print(f"llm_filter: wrote {const.rel(db_path)}")

meta_path = output_dir / "filter_meta.json"
with open(meta_path, "w") as f:
    json.dump(filter_meta, f, indent=2)
print(f"llm_filter: wrote {const.rel(meta_path)}")

## Build filtered and dropped matches

Apply the LLM drop decisions to the cosine match results.
Write both kept and dropped candidates in chunks to avoid accumulating
all rows in memory.

In [ ]:
#|export
filtered_path = output_dir / "filtered_matches.parquet"
dropped_path = output_dir / "dropped_matches.parquet"

failed_set = set(filter_meta["failed_ids"])
successful_ad_ids = [i for i in ad_ids if i not in failed_set]

if filtered_path.exists() and dropped_path.exists():
    # Skip re-export: parquets already exist from a previous run.
    # Re-exporting is non-deterministic (async chunk order) and takes ~35 min.
    _meta = pq.read_metadata(filtered_path)
    total_kept = _meta.num_rows
    _meta_d = pq.read_metadata(dropped_path)
    total_dropped = _meta_d.num_rows
    print(f"llm_filter: parquets already exist, skipping export")
    print(f"llm_filter: {total_kept} kept, {total_dropped} dropped for {len(successful_ad_ids)} ads")
    print(f"  mean candidates kept: {total_kept / max(len(successful_ad_ids), 1):.1f}")
    print(f"  filtered: {filtered_path}")
    print(f"  dropped: {dropped_path}")
else:
    import duckdb
    filter_conn = duckdb.connect(str(db_path))
    filter_rows = filter_conn.execute(
        "SELECT id, data FROM results WHERE error IS NULL"
    ).fetchall()
    filter_conn.close()

    filtered_schema = pa.schema([
        ("ad_id", pa.int64()),
        ("rank", pa.int32()),
        ("onet_code", pa.string()),
        ("onet_title", pa.string()),
        ("cosine_score", pa.float32()),
    ])
    dropped_schema = pa.schema([
        ("ad_id", pa.int64()),
        ("onet_code", pa.string()),
        ("onet_title", pa.string()),
        ("cosine_score", pa.float32()),
        ("original_rank", pa.int32()),
    ])

    filtered_writer = pq.ParquetWriter(filtered_path, filtered_schema)
    dropped_writer = pq.ParquetWriter(dropped_path, dropped_schema)

    # Use the Arrow table + offset index instead of per-chunk DuckDB queries
    _codes = candidates_table.column("onet_code")
    _titles = candidates_table.column("onet_title")
    _scores = candidates_table.column("cosine_score")

    total_kept = 0
    total_dropped = 0

    FILTER_CHUNK_SIZE = 5000
    for chunk_start in range(0, len(filter_rows), FILTER_CHUNK_SIZE):
        chunk = filter_rows[chunk_start:chunk_start + FILTER_CHUNK_SIZE]

        kept_rows = []
        drop_rows = []
        for ad_id_raw, data_str in chunk:
            ad_id = int(ad_id_raw)
            if ad_id not in candidates_index:
                continue
            start, end = candidates_index[ad_id]
            parsed = json.loads(data_str)
            keep_set = set(parsed["keep"])  # 1-based indices

            rank = 0
            for i in range(end - start):
                row_idx = start + i
                code = _codes[row_idx].as_py()
                title = _titles[row_idx].as_py()
                score = float(_scores[row_idx].as_py())
                if (i + 1) in keep_set:
                    kept_rows.append({
                        "ad_id": ad_id,
                        "rank": rank,
                        "onet_code": code,
                        "onet_title": title,
                        "cosine_score": score,
                    })
                    rank += 1
                else:
                    drop_rows.append({
                        "ad_id": ad_id,
                        "onet_code": code,
                        "onet_title": title,
                        "cosine_score": score,
                        "original_rank": i,
                    })

        if kept_rows:
            filtered_writer.write_table(pa.Table.from_pylist(kept_rows, schema=filtered_schema))
            total_kept += len(kept_rows)
        if drop_rows:
            dropped_writer.write_table(pa.Table.from_pylist(drop_rows, schema=dropped_schema))
            total_dropped += len(drop_rows)

    filtered_writer.close()
    dropped_writer.close()

    print(f"llm_filter: {total_kept} kept, {total_dropped} dropped for {len(successful_ad_ids)} ads")
    print(f"  mean candidates kept: {total_kept / max(len(successful_ad_ids), 1):.1f}")
    print(f"  filtered: {filtered_path}")
    print(f"  dropped: {dropped_path}")

successful_ad_ids #|func_return_line